In [ ]:
import os
import sys

if os.path.basename(os.getcwd()) == "analysis":
    os.chdir(os.path.dirname(os.getcwd()))
    sys.path.append(os.getcwd())

full_runs_dir = "../logs_cluster/logs/full_runs/"




In [9]:
from pathlib import Path
import re
import pandas as pd

experiment_dirs = [
    "march/2026_03_10_kodak_2x2_big_arm",
    "january/2026_01_23_YCoCg_big_arm_Kodak",
]

required_text_sections = {
    "processing_image": "Processing image",
    "color_space": "Using color space",
    "image_arm": "Using image ARM",
    "encoder_gain": "Using encoder gain",
    "multi_region_arm": "Using multi-region image ARM",
}

quantized_pattern = re.compile(
    r"Final results after quantization: Loss: ([\d.eE+-]+), "
    r"Rate NN: ([\d.eE+-]+), Rate Latent: ([\d.eE+-]+), Rate Img: ([\d.eE+-]+)"
)

kodim_pattern = re.compile(r"kodim(\d{2})")

rows = []

for exp_dir in experiment_dirs:
    results_dir = Path(full_runs_dir) / exp_dir / "results"

    if not results_dir.exists():
        print(f"[MISSING FOLDER] {results_dir}")
        continue

    log_files = sorted(results_dir.glob("*.log"))
    if not log_files:
        print(f"[NO LOG FILES] {results_dir}")
        continue

    for log_path in log_files:
        text = log_path.read_text(encoding="utf-8", errors="replace")

        missing_sections = [
            name for name, marker in required_text_sections.items() if marker not in text
        ]

        quantized_match = quantized_pattern.search(text)
        if quantized_match is None:
            missing_sections.append("final_quantized_metrics")

        kodim_match = kodim_pattern.search(log_path.name)
        if kodim_match is None:
            missing_sections.append("kodim_two_digit_id")

        if missing_sections:
            print(
                f"[SKIP] {log_path.name} | missing: {', '.join(missing_sections)}"
            )
            continue

        loss, rate_nn, rate_latent, rate_img = map(float, quantized_match.groups())
        kodim_id = int(kodim_match.group(1))

        rows.append(
            {
                "experiment": exp_dir,
                "kodim_id": kodim_id,
                "log_file": log_path.name,
                "loss": loss,
                "rate_nn": rate_nn,
                "rate_latent": rate_latent,
                "rate_img": rate_img,
            }
        )

if not rows:
    print("No valid log files were parsed.")
else:
    df = pd.DataFrame(rows).sort_values(["experiment", "kodim_id"]).reset_index(drop=True)

    # Space-separated table output (easy to copy).
    print(df.to_string(index=False, float_format=lambda x: f"{x:.9f}"))


                            experiment  kodim_id                                                log_file        loss     rate_nn  rate_latent    rate_img
january/2026_01_23_YCoCg_big_arm_Kodak         1        2026_01_28__20_46_56__coolchic_kodak_kodim01.log 3.244920969 0.097128974  0.041012265 3.106779814
january/2026_01_23_YCoCg_big_arm_Kodak         2        2026_01_28__20_43_06__coolchic_kodak_kodim02.log 2.927422047 0.101698981  0.520949483 2.304773569
january/2026_01_23_YCoCg_big_arm_Kodak         3        2026_01_28__21_42_18__coolchic_kodak_kodim03.log 2.413458586 0.103188409  0.128872558 2.181397438
january/2026_01_23_YCoCg_big_arm_Kodak         4        2026_01_28__21_37_43__coolchic_kodak_kodim04.log 3.191143513 0.111504449  0.248532042 2.831106901
january/2026_01_23_YCoCg_big_arm_Kodak         5        2026_01_29__03_56_16__coolchic_kodak_kodim05.log 3.225271940 0.098505656  0.076251447 3.050514698
january/2026_01_23_YCoCg_big_arm_Kodak         6        2026_01_29__03_33_58